In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter, NullFormatter, ScalarFormatter
import numpy as np
import pandas as pd
import seaborn as sb
import seaborn.objects as so
import math
mapping = {
    "faceBased": "Face-Based",
    "globalFaceBased": "Face-Based Global",
    "batchedFace": "Face-Based Batched",
    "cellBased": "Cell-Based",
    "optimal": "Ideal Speedup",
}

df = pd.read_csv("../results/icofoam-h200.csv", skip_blank_lines=True)
cluster = pd.read_csv("../results/clusterCPUScaling.csv", skip_blank_lines=True)


In [ ]:
def setup_df(base_df: pd.DataFrame) -> pd.DataFrame:
    extracted_cells = base_df["case_long"].astype(str).str.extract(r"(\d+)")[0]
    base_df = base_df[extracted_cells.notna()].copy()
    base_df["cells"] = extracted_cells[extracted_cells.notna()].astype(int)
    base_df["time_mean_ms"] = pd.to_numeric(base_df["time_mean_ms"], errors="coerce")
    base_df = base_df[base_df["time_mean_ms"].notna()].copy()
    base_df["ms_normed"] = base_df["time_mean_ms"] / base_df["cells"]
    base_df = base_df[base_df["executor"] == "cpu"]
    base_df["Threads"] = [
        int(x.split("-")[1]) if x.split("-")[1] != "serial" else 1
        for x in base_df["variant"]
    ]
    base_df["cellsPerCpu"] = base_df["cells"] / base_df["Threads"]
    base_df["logth"] = [int(math.log2(x)) for x in base_df["Threads"]]
    group_cols = [
        "case_long",
        "strategy",
        "language",
        "precision",
        "executor",
        "use_kernelAbstractions",
        "use_fusing",
    ]

    baseline = (
        base_df[base_df["variant"].str.contains("serial")]
        .groupby(group_cols, as_index=False)["time_mean_ms"]
        .first()
        .rename(columns={"time_mean_ms": "serial_time"})
    )

    base_df = base_df.merge(baseline, on=group_cols, how="left")
    base_df["speedup"] = base_df["serial_time"] / base_df["time_mean_ms"]
    base_df["efficiency"] = base_df["speedup"] / base_df["Threads"]

    sb.set_theme()
    base_df["logthp1"] = base_df["logth"] + 1
    mapping = {
        "faceBased": "Face-Based",
        "globalFaceBased": "Face-Based Global",
        "batchedFace": "Face-Based Batched",
        "cellBased": "Cell-Based",
        "optimal": "Ideal Speedup",
    }
    bins = [0, 50000, 100000, 300000, 500000, max(base_df["cellsPerCpu"])]
    labels = ["0-50k", "50k-100k", "100k-300k", "300k-500k", "500k+"]

    base_df["CPTranges"] = pd.cut(base_df["cellsPerCpu"], bins=bins, labels=labels)
    base_df["case_num"] = base_df["case_long"].str.extract(r"(\d+)/?$").astype(int)

    # build sorted order
    order = (
        base_df[["case_long", "case_num"]]
        .drop_duplicates()
        .sort_values("case_num")["case_long"]
    )
    ncells = sorted((round(base_df["cells"] ** (1 / 3))).unique().astype(int))
    strats = ["faceBased", "globalFaceBased", "batchedFace", "cellBased"]
    base_df["avg_time"] = base_df.groupby(["Threads", "strategy"])[
         "time_mean_ms"
    ].transform("mean")
    base_df["avg_speedup"] = base_df.groupby(["Threads", "strategy"])[
        "speedup"
    ].transform("mean")
    base_df["avg_efficiency"] = base_df.groupby(["Threads", "strategy"])[
        "efficiency"
    ].transform("mean")
    base_df["avg_normed"] = base_df.groupby(["cells", "strategy"])[
        "ms_normed"
    ].transform("mean")
    base_df["sThreds"] = [f"{x}" for x in base_df["Threads"]]
    base_df["strategy_display"] = [mapping[s] for s in base_df["strategy"]]
    tmap = {np.int64(1): 0,
        np.int64(2): 1,
        np.int64(4): 2,
        np.int64(8): 3,
        np.int64(16): 4,
        np.int64(24): 5,
        np.int64(32): 6,
        np.int64(64): 7,
        np.int64(128): 8,
        np.int64(256): 9}
    base_df["threadindex"] = base_df["Threads"].apply(lambda x: tmap[x]+1)

    return base_df

In [ ]:
julia = pd.read_csv("../results/cpuScaling.csv", skip_blank_lines=True)
julia["Threads"] = [
    int(x.split("-")[1]) if x.split("-")[1] != "serial" else 1
    for x in julia["variant"]
]
julia_serial = julia[julia["Threads"] == 1]
julia_serial["cells"] = [int(x.rsplit("/", 2)[1]) ** 3 for x in julia_serial["case_long"]]
julia_serial["ms_normed"] = julia_serial["time_mean_ms"] / julia_serial["cells"]
julia_serial["time_ms"] = julia_serial["time_mean_ms"]

julia_serial["strategy_display"] = [mapping[s] for s in julia_serial["strategy"]]


In [ ]:
julia

In [ ]:
cluster = setup_df(cluster)


In [ ]:

p = sb.relplot(data=df, x="cells", y="ms_normed", height=10, hue="np_string", kind="line")
axes = p.axes.flat
# for ax in axes:
#     sb.scatterplot(ax=ax, data=julia_serial, x="cells", y="ms_normed")
#     ax.set_xscale("log")
p.set(
    yscale="log",
)

In [ ]:
all = pd.concat([julia, df])

In [ ]:
sb.set_style("whitegrid")
sb.color_palette("deep", 8)
plot =sb.relplot(data=all[all["Threads"] == 1], x="cells", y="ms_normed", hue="strategy_display", height=10, s=80, style="strategy_display")
plot.set(
    yscale="log",
    xscale="log"
)
plot.legend.set_title("Strategy")

In [ ]:
sb.set_style("whitegrid")
sb.color_palette("deep", 8)
plot =sb.relplot(data=all, x="cells", y="time_ms", hue="strategy", height=10, s=80, style="strategy")
plot.set(
    yscale="log",
    xscale="log"
)

In [ ]:
def setup_icofoam(base_df: pd.DataFrame) -> pd.DataFrame:
    base_df["time_ms"] = base_df["time_s"] * 1000
    if "cell_dim" in base_df.columns:
        base_df["cells"] = base_df["cell_dim"].apply(lambda x: x**3)
    else:
        base_df["cells"] = pd.to_numeric(base_df["cells"], errors="raise").astype(int)
    base_df["ms_normed"] = base_df["time_ms"] / base_df["cells"]
    base_df["time_mean_ms"] = base_df["time_ms"]
    base_df["strategy"] = "IcoFoam"
    base_df["strategy_display"] = "IcoFoam H200"
    base_df["np_string"] = base_df["nprocs"].apply(lambda x: str(x))
    base_df["Threads"] = base_df["nprocs"]
    base_df["ms_normed"] = base_df["time_mean_ms"] / base_df["cells"]
    base_df["cellsPerCpu"] = base_df["cells"] / base_df["Threads"]
    base_df["logth"] = [int(math.log2(x)) for x in base_df["Threads"]]

    baseline_nprocs = int(base_df["nprocs"].min())
    baseline = (
        base_df[base_df["nprocs"] == baseline_nprocs]
        .groupby(["cells"], as_index=False)["time_ms"]
        .first()
        .rename(columns={"time_ms": "baseline_time"})
    )

    base_df = base_df.merge(baseline, on=["cells"], how="left")
    base_df["serial_time"] = base_df["baseline_time"] * baseline_nprocs
    base_df["speedup"] = base_df["serial_time"] / base_df["time_mean_ms"]
    base_df["efficiency"] = base_df["speedup"] / base_df["Threads"]

    sb.set_theme()
    base_df["logthp1"] = base_df["logth"] + 1
    mapping = {
        "faceBased": "Face-Based",
        "globalFaceBased": "Face-Based Global",
        "batchedFace": "Face-Based Batched",
        "cellBased": "Cell-Based",
        "optimal": "Ideal Speedup",
    }
    bins = [0, 50000, 100000, 300000, 500000, max(base_df["cellsPerCpu"])]
    labels = ["0-50k", "50k-100k", "100k-300k", "300k-500k", "500k+"]

    base_df["CPTranges"] = pd.cut(base_df["cellsPerCpu"], bins=bins, labels=labels)
    base_df["avg_time"] = base_df.groupby(["Threads"])[
         "time_mean_ms"
    ].transform("mean")
    base_df["avg_speedup"] = base_df.groupby(["Threads"])[
        "speedup"
    ].transform("mean")
    base_df["avg_efficiency"] = base_df.groupby(["Threads"])[
        "efficiency"
    ].transform("mean")
    base_df["avg_normed"] = base_df.groupby(["cells", "strategy"])[
        "ms_normed"
    ].transform("mean")
    base_df["sThreds"] = [f"{x}" for x in base_df["Threads"]]
    tmap = {np.int64(1): 0,
        np.int64(2): 1,
        np.int64(4): 2,
        np.int64(8): 3,
        np.int64(16): 4,
        np.int64(24): 5,
        np.int64(32): 6,
        np.int64(64): 7,
        np.int64(128): 8,
        np.int64(256): 9}
    base_df["threadindex"] = base_df["Threads"].apply(lambda x: tmap[x]+1)

    return base_df

In [ ]:
df =setup_icofoam(df)

In [ ]:
all = pd.concat([cluster, df])


In [ ]:
def scaling_ms(data):
    sb.set_style("whitegrid")
    sb.color_palette("deep", 8)

    with sb.plotting_context("paper", font_scale=1.5):
        plot = sb.relplot(data=data, x="logthp1", y="time_mean_ms", kind="scatter", hue="np_string",s=60)
        axes = plot.axes.flat

        for i, ax in enumerate(axes):
            sb.lineplot(
                data=data,
                x="logthp1",
                y="time_mean_ms",
                hue="strategy",
                legend=False,   # avoid duplicate legends
                ax=ax
            )
            ax.tick_params(labelbottom=True)
            ax.set_xticks(list(range(1,1+len(data["Threads"].unique()))))
            ax.set_xticklabels(labels=data["Threads"].unique())
            ax.set_xlabel("Threads")
            ax.plot([1,9], [1,9], color='k',linestyle="dashed", label="Ideal Speedup")
            ax.set_yticks(list(range(1,10)))

            # plot.legend(False)
        # sb.move_legend(plot, "upper left", bbox_to_anchor=(0.1 ,0.9))
        handles, labels = axes[0].get_legend_handles_labels()
        plot.legend.remove()
        # plot.set(
        #     ylabel="Speedup"
        # )
        plt.ylabel("Speedup", fontsize=30)
        axes[0].legend(handles, labels, loc="upper left",
                        title="Strategy")

        plt.tight_layout()
scaling_ms(df)


In [ ]:
scaling_ms(df)

In [ ]:
all["max_speedup"] = all.groupby(["Threads", "cells"])[
    "speedup"
].transform("max")

In [ ]:
all["min_times"] = all.groupby(["Threads", "cells", "strategy"])[
    "ms_normed"
].transform("min")

In [ ]:
def avgnormed(df, y):
    sb.set_style("whitegrid")
    sb.color_palette("deep", 8)
    with sb.plotting_context("paper", font_scale=1.7):
        plot = sb.relplot(data=df, x="cells", y=y, hue="strategy_display", kind="scatter", height=8, s=60)
        axes = plot.axes.flat
        for ax in axes:
            sb.lineplot(
                data=df,
                x="cells",
                y=y,
                hue="strategy_display",
                legend=False,
                ax=ax,
            )

            # Ideal scaling: speedup = threads → log2(s) = log2(t)
            x_vals = sorted(df["logthp1"].unique())

            threads = sorted(df[df["Threads"]< 256]["Threads"].unique())
            
            ax.set_xticks(x_vals)
            ax.set_xticklabels(threads)

            ax.set_yticks(x_vals)
            ax.set_yticklabels(threads)
        plot.set(
            xscale="log",
            yscale="log"
        )
        handles, _ = axes[0].get_legend_handles_labels()

        plot.legend.remove()

        axes[0].legend(
            handles,
            df["strategy_display"].unique(),
            loc="upper right",
            title="Strategy"
        )
        plt.ylabel("Assembly time per cell (ms)", fontsize=20)
        plt.xlabel("Cells", fontsize=20)
        plt.tight_layout()
        output_name = "min_times" if y == "ms_normed" else y
        plt.savefig(f"../figures/icoFoam-julia-{output_name}.svg")
avgnormed(all[all["cells"]<=8000000], "ms_normed")

In [ ]:
best = (
    all.loc[
        all.groupby(["cells", "strategy"])["time_mean_ms"].idxmin()
    ]
    .sort_values(["cells", "strategy"])
    .reset_index(drop=True)
)


In [ ]:
best["ms_normed"] = best["time_mean_ms"] / best["cells"]

In [ ]:
p = sb.relplot(data=best, x="cells", y="ms_normed", kind="line", hue="strategy", height=8)
p.set(yscale="log", xscale="log")

In [ ]:
best

In [ ]:
sorted(cluster["cells"].unique())

In [ ]:
sorted(all["logthp1"].unique())

In [ ]:
all["log_speedup"] = np.log2(all["avg_speedup"])


In [ ]:
scaling_speedup(all[(all["Threads"] < 256) & (all["cells"] == 8000000)])

In [ ]:
def scaling_efficiency(data):
    sb.set_style("whitegrid")
    sb.color_palette("deep", 8)
    with sb.plotting_context("paper", font_scale=1.5):

        plot = sb.relplot(data=data, x="threadindex", y="avg_efficiency", kind="scatter", style="strategy", hue="strategy", s=60)
        axes = plot.axes.flat

        for i, ax in enumerate(axes):
            sb.lineplot(
                data=data,
                x="threadindex",
                y="avg_efficiency",
                hue="strategy",
                legend=False,   # avoid duplicate legends
                ax=ax
            )
            ax.tick_params(labelbottom=True)
            ax.set_xticks(list(range(1,9)))
            ax.set_xticklabels(labels=sorted(data["Threads"].unique()))
            ax.set_xlabel("Threads")
        handles, labels = axes[0].get_legend_handles_labels()
        sb.move_legend(plot, "upper right", fancybox=True)
        plot.legend.remove()
        plt.ylabel("Efficiency", fontsize=20)
        plt.xlabel("Threads", fontsize=20)
        axes[0].legend(handles,  all["strategy_display"].unique(), loc="upper right",
                        title="Strategy")

        plt.tight_layout()
        plt.savefig("../figures/icoFoam-julia-efficiency.svg")

scaling_efficiency(all[all["Threads"] < 256])

In [ ]:
sorted(all["cells"].unique())

In [ ]:
def scaling_ms(data):
    sb.set_style("whitegrid")
    sb.color_palette("deep", 8)
    with sb.plotting_context("paper", font_scale=1.5):

        plot = sb.relplot(data=data, x="cells", y="avg_time", kind="scatter", style="strategy", hue="strategy", s=60)
        axes = plot.axes.flat
        plot.set(
            xscale="log",
            yscale="log"
        )
        for i, ax in enumerate(axes):
            sb.lineplot(
                data=data,
                x="cells",
                y="avg_time",
                hue="strategy",
                legend=False,   # avoid duplicate legends
                ax=ax,
                err_style=None
            )
            # ax.tick_params(labelbottom=True)
            # ax.set_xticks(list(range(1,11)))
            # ax.set_xticklabels(labels=sorted(data["Threads"].unique()))
            # ax.set_xlabel("Threads")
        handles, labels = axes[0].get_legend_handles_labels()
        sb.move_legend(plot, "upper right", fancybox=True)
        plot.legend.remove()
        plt.ylabel("Efficiency", fontsize=20)
        # plt.xlabel("Threads", fontsize=20)
        axes[0].legend(handles,  all["strategy_display"].unique(), loc="upper right",
                        title="Strategy")

        plt.tight_layout()
scaling_ms(all)